# Phase 3 – Classification: Bubble Detection & Relative Row Scoring

This notebook demonstrates:

1. **BubbleClassifier** – a lightweight CNN that classifies individual bubbles
   as *filled* or *empty*.
2. **RelativeRowScorer** – determines the selected answer per row by comparing
   fill probabilities *relative to each other*.

> ⚠️ **Critical design principle**: Phase 3 uses **Relative Row Scoring**, NOT
> a flat darkness threshold. This makes grading robust to global exposure
> variation across the sheet.

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Add Phase 3 src to the Python path.
phase3_src = Path("../Phase_3_Classification/src").resolve()
if str(phase3_src) not in sys.path:
    sys.path.insert(0, str(phase3_src))

print(f"Phase 3 src path: {phase3_src}")
print(f"Path exists: {phase3_src.exists()}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from bubble_classifier import BubbleClassifier
from relative_row_scorer import RelativeRowScorer

print(f"BubbleClassifier  : {BubbleClassifier}")
print(f"RelativeRowScorer : {RelativeRowScorer}")
print(f"PyTorch version   : {torch.__version__}")

## 2. BubbleClassifier – Architecture

```
Input: (N, 3, H, W)  float32 tensor

  Conv2d(3→32, 3×3)  → BN → ReLU → MaxPool(2×2)
  Conv2d(32→64, 3×3) → BN → ReLU → MaxPool(2×2)
  Conv2d(64→128,3×3) → BN → ReLU → MaxPool(2×2)
  AdaptiveAvgPool(1×1)
  Flatten
  Linear(128, num_classes)

Output: (N, num_classes)  raw logits
```

Three inference methods:
- `forward(x)` → raw logits
- `predict_proba(x)` → softmax probabilities
- `predict(x)` → argmax class indices

In [ ]:
model = BubbleClassifier(num_classes=2)
model.eval()
print(model)

# Count trainable parameters.
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

## 3. Running Predictions on Synthetic Bubble Images

We create synthetic 32×32 bubble crops:
- **Filled** bubble: dark circle on light background
- **Empty** bubble: light circle on light background

In [ ]:
import cv2

def make_bubble(filled: bool, size: int = 32) -> np.ndarray:
    """Create a synthetic bubble image (BGR uint8)."""
    img = np.ones((size, size, 3), dtype=np.uint8) * 230
    colour = (30, 30, 30) if filled else (210, 210, 210)
    cx, cy, r = size // 2, size // 2, size // 2 - 3
    cv2.circle(img, (cx, cy), r, colour, -1)
    cv2.circle(img, (cx, cy), r, (80, 80, 80), 1)
    return img

def bgr_to_tensor(img: np.ndarray) -> torch.Tensor:
    """BGR uint8 HWC → float32 CHW [0,1] tensor."""
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    chw = np.transpose(rgb, (2, 0, 1)).astype(np.float32) / 255.0
    return torch.from_numpy(chw)

# Build a batch: [filled, empty, filled, empty, empty]
labels_gt = [1, 0, 1, 0, 0]
bubbles = [make_bubble(filled=(lbl == 1)) for lbl in labels_gt]
tensors = torch.stack([bgr_to_tensor(b) for b in bubbles])  # (5, 3, 32, 32)

# Display the synthetic bubbles.
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, (bubble, lbl) in enumerate(zip(bubbles, labels_gt)):
    axes[i].imshow(cv2.cvtColor(bubble, cv2.COLOR_BGR2RGB))
    axes[i].set_title("Filled" if lbl else "Empty")
    axes[i].axis("off")
plt.suptitle("Synthetic Bubble Crops (input to BubbleClassifier)")
plt.tight_layout()
plt.show()

print(f"Batch tensor shape: {tensors.shape}")

In [ ]:
# Run BubbleClassifier inference (untrained model = random weights; for demo only).
with torch.no_grad():
    logits = model(tensors)
    proba  = model.predict_proba(tensors)
    preds  = model.predict(tensors)

print("Logits  (raw):\n", logits.numpy().round(4))
print("\nProbabilities (softmax):\n", proba.numpy().round(4))
print("\nPredictions (argmax):", preds.numpy())
print("Ground truth        :", labels_gt)
print("\nNote: model is UNTRAINED – probabilities are random. Real use requires training.")

## 4. Relative Row Scoring

### Why NOT a flat threshold?

A flat threshold (e.g., "bubble is filled if darkness > 0.6") fails when:
- The sheet is uniformly dark (all bubbles appear filled)
- The sheet is uniformly light (all bubbles appear empty even when one is filled)
- There is a shadow gradient across rows

### Relative Row Scoring algorithm

For each answer row:
1. Find the bubble with the **highest fill probability** → candidate.
2. If candidate probability < `min_fill_prob` (0.3) → **blank row**.
3. If top-2 probabilities differ by < `tie_threshold` (0.15) → **erasure tie** (ambiguous).
4. Otherwise → **selected** = candidate label.
5. **Confidence** = top probability − mean of all other probabilities.

In [ ]:
scorer = RelativeRowScorer(tie_threshold=0.15, min_fill_prob=0.3)
print(f"tie_threshold : {scorer.tie_threshold}")
print(f"min_fill_prob : {scorer.min_fill_prob}")

In [ ]:
# --- Demonstrate score_row with hand-crafted probability scenarios -----------

labels = ["A", "B", "C", "D", "E"]

scenarios = [
    # name,                   probs for A, B, C, D, E
    ("Clear answer C",        [0.05, 0.08, 0.82, 0.03, 0.02]),
    ("Tie between A and B",   [0.48, 0.46, 0.02, 0.02, 0.02]),
    ("Blank row",             [0.10, 0.12, 0.08, 0.09, 0.11]),
    ("Dark sheet (all high)", [0.71, 0.65, 0.62, 0.64, 0.63]),  # D selected, not flat threshold
    ("Clear answer E",        [0.04, 0.03, 0.05, 0.07, 0.81]),
]

print(f"{'Scenario':<30}  {'Selected':>8}  {'is_blank':>8}  {'is_tie':>8}  {'confidence':>10}")
print("-" * 72)

for name, probs in scenarios:
    result = scorer.score_row(probs, labels)
    sel  = result["selected"] or "—"
    blank = result["is_blank"]
    tie   = result["is_tie"]
    conf  = result["confidence"]
    print(f"{name:<30}  {sel:>8}  {str(blank):>8}  {str(tie):>8}  {conf:>10.4f}")

### Key observation – Relative scoring on the "Dark sheet" scenario

In the *"Dark sheet (all high)"* scenario, all five probabilities are above 0.60.
A flat threshold of 0.60 would select **all five** bubbles as filled.
**Relative Row Scoring correctly selects only bubble A** (0.71) as the answer
because it is the most filled *relative to its neighbours*.

In [ ]:
# Visual comparison: flat threshold vs relative scoring.
dark_probs  = [0.71, 0.65, 0.62, 0.64, 0.63]
FLAT_THRESHOLD = 0.60

flat_selections   = [lbl for lbl, p in zip(labels, dark_probs) if p >= FLAT_THRESHOLD]
relative_result   = scorer.score_row(dark_probs, labels)
relative_selection = relative_result["selected"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colours_flat = ["tomato" if lbl in flat_selections else "steelblue" for lbl in labels]
colours_rel  = ["tomato" if lbl == relative_selection else "steelblue" for lbl in labels]

axes[0].bar(labels, dark_probs, color=colours_flat)
axes[0].axhline(FLAT_THRESHOLD, color="red", linestyle="--", label=f"Flat threshold = {FLAT_THRESHOLD}")
axes[0].set_title(f"Flat Threshold (selects: {flat_selections})")
axes[0].set_ylabel("Fill probability")
axes[0].set_ylim(0, 1)
axes[0].legend()

axes[1].bar(labels, dark_probs, color=colours_rel)
axes[1].set_title(f"Relative Row Scoring (selects: {relative_selection})")
axes[1].set_ylabel("Fill probability")
axes[1].set_ylim(0, 1)

plt.suptitle("Flat Threshold vs. Relative Row Scoring on Uniformly Dark Sheet", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Flat threshold selections : {flat_selections}  ← WRONG (multiple selections)")
print(f"Relative Row Scoring      : {relative_selection}  ← CORRECT (single selection)")

## 5. Full Sheet Scoring with `score_sheet`

`score_sheet()` applies `score_row()` to every row of an answer sheet and
returns a list of result dicts.

In [ ]:
# Simulate a 10-question answer sheet (A–E bubbles per row).
sheet_probs = [
    [0.05, 0.08, 0.82, 0.03, 0.02],   # Q1 → C
    [0.81, 0.06, 0.05, 0.04, 0.04],   # Q2 → A
    [0.04, 0.85, 0.04, 0.04, 0.03],   # Q3 → B
    [0.48, 0.46, 0.02, 0.02, 0.02],   # Q4 → tie A/B
    [0.10, 0.12, 0.08, 0.09, 0.11],   # Q5 → blank
    [0.03, 0.02, 0.03, 0.88, 0.04],   # Q6 → D
    [0.02, 0.04, 0.03, 0.05, 0.86],   # Q7 → E
    [0.71, 0.65, 0.62, 0.64, 0.63],   # Q8 → A (dark sheet)
    [0.06, 0.84, 0.04, 0.03, 0.03],   # Q9 → B
    [0.03, 0.04, 0.87, 0.03, 0.03],   # Q10 → C
]
sheet_labels = [["A", "B", "C", "D", "E"]] * 10

results = scorer.score_sheet(sheet_probs, sheet_labels)

print(f"{'Q':>3}  {'Selected':>8}  {'is_blank':>8}  {'is_tie':>8}  {'confidence':>10}")
print("-" * 50)
for q, res in enumerate(results, start=1):
    sel  = res["selected"] or "—"
    blank = res["is_blank"]
    tie   = res["is_tie"]
    conf  = res["confidence"]
    print(f"Q{q:<2}  {sel:>8}  {str(blank):>8}  {str(tie):>8}  {conf:>10.4f}")

## 6. End-to-End Integration Sketch

How Phase 3 connects to Phases 1 & 2:

In [ ]:
# Pseudocode sketch – replace with real paths/model to run end-to-end.

integration_sketch = """
# ---- Phase 1: Form Extraction ----
detector  = YOLODetector("omr_detector.pt", conf_threshold=0.5)
extractor = FormExtractor(detector, target_size=(800, 1000))
form_img  = extractor.extract(raw_photo)         # np.ndarray (1000, 800, 3)

# ---- Phase 2: Preprocessing ----
prep    = CLAHEPreprocessor(clip_limit=2.0)
cleaned = prep.process(form_img)                 # CLAHE-normalised BGR image

# ---- Phase 3: Bubble classification (per-crop) ----
bubble_classifier = BubbleClassifier(num_classes=2)
# ... load trained weights: bubble_classifier.load_state_dict(...)

# For each detected bubble crop in the form:
row_probs  = []  # fill probabilities per row
for row_crops in detected_bubble_rows:
    batch  = torch.stack([prep.to_tensor(c) for c in row_crops])
    proba  = bubble_classifier.predict_proba(batch)  # (5, 2)
    fill_p = proba[:, 1].tolist()                    # probability of class 1 = filled
    row_probs.append(fill_p)

# ---- Phase 3: Relative Row Scoring ----
scorer  = RelativeRowScorer(tie_threshold=0.15, min_fill_prob=0.3)
answers = scorer.score_sheet(row_probs, row_labels)
"""

print(integration_sketch)

## Summary

| Component | Role |
|---|---|
| `BubbleClassifier` | CNN classifies each bubble as filled (1) or empty (0) |
| `predict_proba()` | Returns softmax fill probability for each bubble |
| `RelativeRowScorer` | Selects the answer per row via **relative comparison** |
| `score_row()` | Single-row scoring with blank + tie detection |
| `score_sheet()` | Applies `score_row()` to all rows of an answer sheet |

> **Relative Row Scoring is the key innovation**: instead of asking if a bubble
> is dark enough; it asks which bubble is darkest relative to its neighbours.
> This makes the system robust to global illumination variation.